In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load cleaned dataset
df = pd.read_csv("../data/telco_cleaned.csv")

# Separate features and target
X = df.drop("Churn", axis=1)
y = df["Churn"].map({"No": 0, "Yes": 1})

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Identify feature types
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Preprocessing
preprocessor = ColumnTransformer([
    (
        "num",
        StandardScaler(),
        numerical_features
    ),
    (
        "cat",
        OneHotEncoder(
            handle_unknown="ignore",
            drop="first"
        ),
        categorical_features
    )
])

print("Data loaded and preprocessing setup complete.")
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Data loaded and preprocessing setup complete.
Training shape: (5616, 24)
Testing shape: (1405, 24)


C:\Users\sansk\AppData\Local\Temp\ipykernel_20776\2209165456.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [3]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

logistic_params = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"]
}

logistic_grid = GridSearchCV(
    logistic_pipeline,
    logistic_params,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

logistic_grid.fit(X_train, y_train)

print("Best parameters:")
print(logistic_grid.best_params_)

print("Best CV ROC-AUC:")
print(logistic_grid.best_score_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters:
{'model__C': 10, 'model__solver': 'liblinear'}
Best CV ROC-AUC:
0.8465234690389938


In [4]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_params = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [3, 4, 5],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

In [5]:
from sklearn.model_selection import RandomizedSearchCV

xgb_random = RandomizedSearchCV(
    xgb_pipeline,
    xgb_params,
    n_iter=30,
    cv=5,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_random.fit(X_train, y_train)

print("Best parameters:")
print(xgb_random.best_params_)

print("Best CV ROC-AUC:")
print(xgb_random.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters:
{'model__subsample': 0.8, 'model__n_estimators': 200, 'model__max_depth': 3, 'model__learning_rate': 0.03, 'model__colsample_bytree': 0.8}
Best CV ROC-AUC:
0.8494923198784408


In [6]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

tuned_models = {
    "Tuned Logistic Regression": logistic_grid.best_estimator_,
    "Tuned XGBoost": xgb_random.best_estimator_
}

results = []

for name, model in tuned_models.items():

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

tuned_results = pd.DataFrame(results)

tuned_results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Tuned Logistic Regression,0.741637,0.507881,0.779570,0.615058,0.839994
1,Tuned XGBoost,0.807117,0.678445,0.516129,0.586260,0.843502


In [7]:
best_xgb = xgb_random.best_estimator_

y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]

In [8]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

threshold_results = []

thresholds = np.arange(0.20, 0.71, 0.05)

for threshold in thresholds:

    y_pred_threshold = (
        y_prob_xgb >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            y_test,
            y_pred_threshold
        ),
        "Precision": precision_score(
            y_test,
            y_pred_threshold
        ),
        "Recall": recall_score(
            y_test,
            y_pred_threshold
        ),
        "F1": f1_score(
            y_test,
            y_pred_threshold
        )
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.round(3)

,Threshold,Accuracy,Precision,Recall,F1
0,0.20,0.703,0.467,0.860,0.605
1,0.25,0.735,0.500,0.820,0.621
2,0.30,0.760,0.533,0.750,0.623
3,0.35,0.779,0.569,0.685,0.622
4,0.40,0.794,0.606,0.632,0.618
5,0.45,0.809,0.657,0.578,0.615
6,0.50,0.807,0.678,0.516,0.586
7,0.55,0.796,0.682,0.433,0.530
8,0.60,0.787,0.721,0.320,0.443
9,0.65,0.779,0.733,0.258,0.382


In [9]:
best_threshold_row = threshold_df.loc[
    threshold_df["F1"].idxmax()
]

best_threshold_row

Threshold    0.300000
Accuracy     0.760142
Precision    0.533461
Recall       0.750000
F1           0.623464
Name: 2, dtype: float64

In [10]:
xgb_random.best_params_

{'model__subsample': 0.8,
 'model__n_estimators': 200,
 'model__max_depth': 3,
 'model__learning_rate': 0.03,
 'model__colsample_bytree': 0.8}